# SuperMarioBros PPO Training on Kaggle\n\n这个 notebook 用于在 Kaggle 上训练 `shuishen49/SuperMarioBros-AI` 项目。\n\n## 使用前准备\n1. 在 Kaggle Notebook 设置里打开 **GPU**\n2. 在右侧 Add Data 上传一个数据集，里面包含 ROM zip（例如 `Super Mario Bros. (World).zip`）\n3. 修改下方 `ROM_ZIP_PATH` 为你的实际路径\n

In [ ]:
import os, sys, subprocess\n\ndef run(cmd):\n    print('>>>', cmd)\n    return subprocess.run(cmd, shell=True, check=False)\n\n# Kaggle Linux 环境下可用\nrun('nvidia-smi')\nrun(f'{sys.executable} -V')\n

In [ ]:
# 1) 拉取仓库\nWORKDIR = '/kaggle/working'\nREPO_DIR = os.path.join(WORKDIR, 'SuperMarioBros-AI')\n\nif not os.path.exists(REPO_DIR):\n    run(f'git clone https://github.com/shuishen49/SuperMarioBros-AI.git {REPO_DIR}')\n\n%cd /kaggle/working/SuperMarioBros-AI\nrun('ls -la')\n

In [ ]:
# 2) 安装依赖\n# Kaggle 通常已有 torch+cuda，这里按项目需求补齐\nrun(f'{sys.executable} -m pip install -U pip')\nrun(f'{sys.executable} -m pip install -r requirements.txt')\nrun(f'{sys.executable} -m pip install "gym==0.23.1" "shimmy>=2.0"')\n

In [ ]:
# 3) 导入 ROM 到 gym-retro\n# TODO: 改成你的 Kaggle 数据集 ROM 文件路径\nROM_ZIP_PATH = '/kaggle/input/smb-rom/Super Mario Bros. (World).zip'\n\nassert os.path.exists(ROM_ZIP_PATH), f'ROM 文件不存在: {ROM_ZIP_PATH}'\nrun('python -m retro.import "{}"'.format(ROM_ZIP_PATH))\n

In [ ]:
# 4) 快速环境自检\nimport retro\nenv = retro.make(game='SuperMarioBros-Nes')\nobs = env.reset()\nprint('obs shape =', getattr(obs, 'shape', None))\nenv.close()\n

In [ ]:
# 5) 开始训练（Kaggle 无桌面，禁用 render）\n# 可以改 timesteps，例如 100000 / 300000\nTIMESTEPS = 100000\nrun(f'python train_ppo.py --timesteps {TIMESTEPS} --device cuda')\n

In [ ]:
# 6) 打包模型，便于从 Kaggle 下载\nrun('ls -lah models || true')\nrun('mkdir -p /kaggle/working/output_models')\nrun('cp -r models/* /kaggle/working/output_models/ || true')\nrun('ls -lah /kaggle/working/output_models')\n